# Bronze para Silver: Unidades Consumidoras (V-Energy Grid 360)

Versao 2. Execute as celulas em ordem, de cima para baixo.

Fluxo: leitura da Bronze, rejeicao (quarentena) de lixo e dados de teste, deduplicacao deterministica, validacao estrutural de documentos, padronizacao e tipagem, gravacao da Silver e da quarentena, reconciliacao de volumes e validacoes sobre as tabelas gravadas.

A camada Gold foi separada em outro notebook (silver-to-gold-resumo-ucs).

In [ ]:
# ==============================================================================
# CELULA 1: Configuracao, parametros de regra e leitura da Bronze
# ==============================================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import BooleanType

# Tabelas
bronze_table = "workspace.default.bronze_unidades_consumidoras"
silver_table = "workspace.default.silver_unidades_consumidoras"
quarantine_table = "workspace.default.silver_unidades_consumidoras_quarentena"

# Parametros de regra (documentados na arquitetura)
MIN_DIGITOS_DOCUMENTO = 9            # abaixo disso o documento e considerado incompleto
DATA_LIGACAO_MIN = "1900-01-01"      # datas fora de [MIN, hoje] viram NULL (sentinelas/futuro)
BRASIL_LAT_MIN, BRASIL_LAT_MAX = -34.0, 6.0
BRASIL_LON_MIN, BRASIL_LON_MAX = -74.0, -28.0
CEPS_SENTINELA = ["99991231", "00000000"]
UFS_VALIDAS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

MOTIVOS_PRE = ["id_uc_nulo", "cabecalho_repetido", "dado_de_teste_id_uc",
               "nome_titular_nulo", "dado_de_teste_titular"]
MOTIVOS_POS = ["documento_ausente", "documento_incompleto",
               "documento_tamanho_invalido", "documento_sentinela_repetido"]

def fmt(n):
    return f"{n:,}".replace(",", ".")

# Leitura da Bronze
df_bronze = spark.table(bronze_table)

colunas_esperadas = [
    "id_uc", "cpf_cnpj", "nome_titular", "logradouro", "numero", "bairro", "municipio",
    "uf", "cep", "classe", "subgrupo_tarifario", "alimentador", "data_ligacao", "status",
    "latitude", "longitude", "ingestion_datetime", "source_file",
]
faltantes = [c for c in colunas_esperadas if c not in df_bronze.columns]
assert not faltantes, f"Colunas ausentes na Bronze: {faltantes}"

total_bronze = df_bronze.count()
print(f"Origem Bronze: {bronze_table}")
print(f"Destino Silver: {silver_table}")
print(f"Destino Quarentena: {quarantine_table}")
print(f"Total de registros na Bronze: {fmt(total_bronze)}")
df_bronze.printSchema()

In [ ]:
# ==============================================================================
# CELULA 2: Funcoes auxiliares e validacao de digitos verificadores
# ==============================================================================
ACENTOS_ORIGEM = "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ"
ACENTOS_DESTINO = "AAAAAEEEEIIIIOOOOOUUUUCN"
assert len(ACENTOS_ORIGEM) == len(ACENTOS_DESTINO)

def limpar_espacos(c):
    """trim + colapsa espacos repetidos."""
    return F.regexp_replace(F.trim(c), r"\s+", " ")

def texto_padrao(c):
    """Sem acentos, espacos colapsados e Title Case (usado em chaves de agrupamento)."""
    return F.initcap(F.translate(F.upper(limpar_espacos(c)), ACENTOS_ORIGEM, ACENTOS_DESTINO))

def _cpf_dv_valido(d):
    if d is None or len(d) != 11 or not d.isdigit() or d == d[0] * 11:
        return False
    for n in (9, 10):
        soma = sum(int(d[i]) * (n + 1 - i) for i in range(n))
        dv = (soma * 10) % 11
        dv = 0 if dv == 10 else dv
        if dv != int(d[n]):
            return False
    return True

def _cnpj_dv_valido(d):
    if d is None or len(d) != 14 or not d.isdigit() or d == d[0] * 14:
        return False
    pesos1 = [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
    pesos2 = [6] + pesos1
    for pesos, pos in ((pesos1, 12), (pesos2, 13)):
        soma = sum(int(d[i]) * pesos[i] for i in range(pos))
        resto = soma % 11
        dv = 0 if resto < 2 else 11 - resto
        if dv != int(d[pos]):
            return False
    return True

@F.udf(returnType=BooleanType())
def udf_documento_dv_valido(doc):
    if doc is None:
        return False
    if len(doc) == 11:
        return _cpf_dv_valido(doc)
    if len(doc) == 14:
        return _cnpj_dv_valido(doc)
    return False

In [ ]:
# ==============================================================================
# CELULA 3: Rejeicao de lixo/testes e deduplicacao deterministica
# ==============================================================================
# 3.1 Normalizacao minima das chaves usadas nas regras (as colunas originais ficam intactas)
df_base = (
    df_bronze
    .withColumn("id_uc_norm", F.upper(F.trim(F.col("id_uc"))))
    .withColumn("nome_norm", limpar_espacos(F.col("nome_titular")))
)

# 3.2 Regras de rejeicao (null-safe: nulos tambem vao para a quarentena, nunca somem em silencio)
motivo_pre = (
    F.when(F.col("id_uc_norm").isNull() | (F.col("id_uc_norm") == ""), F.lit("id_uc_nulo"))
    .when(F.col("id_uc_norm") == "ID_UC", F.lit("cabecalho_repetido"))
    .when(F.col("id_uc_norm").startswith("TESTE"), F.lit("dado_de_teste_id_uc"))
    .when(F.col("nome_norm").isNull() | (F.col("nome_norm") == ""), F.lit("nome_titular_nulo"))
    .when(F.lower(F.col("nome_norm")).isin("a", "teste"), F.lit("dado_de_teste_titular"))
)

df_pre = df_base.withColumn("motivo_rejeicao", motivo_pre)
df_rej_pre = df_pre.filter(F.col("motivo_rejeicao").isNotNull())
df_valid_pre = df_pre.filter(F.col("motivo_rejeicao").isNull()).drop("motivo_rejeicao")

# 3.3 Data de ligacao tipada ANTES da dedup (ordenar string e fragil) e restrita a [MIN, hoje]
data_cast = F.expr("try_cast(data_ligacao as date)")
data_valida = F.when(
    (data_cast >= F.lit(DATA_LIGACAO_MIN).cast("date")) & (data_cast <= F.current_date()),
    data_cast,
)

# 3.4 Hash do conteudo de negocio: ultimo criterio de desempate (torna a dedup 100% deterministica)
colunas_negocio = [c for c in df_bronze.columns if c not in ("ingestion_datetime", "source_file")]
row_hash = F.sha2(
    F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in colunas_negocio]),
    256,
)

df_valid_pre = (
    df_valid_pre
    .withColumn("data_ligacao_dt", data_valida)
    .withColumn("row_hash", row_hash)
)

# 3.5 Dedup: versao mais recente por id_uc
window_spec = Window.partitionBy("id_uc_norm").orderBy(
    F.col("ingestion_datetime").desc_nulls_last(),
    F.col("data_ligacao_dt").desc_nulls_last(),
    F.col("source_file").desc_nulls_last(),
    F.col("row_hash").desc(),
)

df_dedup = (
    df_valid_pre
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)
print("Rejeicao e deduplicacao definidas (execucao lazy, nada foi gravado ainda).")

In [ ]:
# ==============================================================================
# CELULA 4: Validacao estrutural de documentos e montagem da quarentena
# ==============================================================================
doc_digitos = F.regexp_replace(F.coalesce(F.col("cpf_cnpj"), F.lit("")), r"\D", "")

motivo_pos = (
    F.when(F.length("doc_digitos") == 0, F.lit("documento_ausente"))
    .when(F.length("doc_digitos") > 14, F.lit("documento_tamanho_invalido"))
    .when(F.length("doc_digitos") < MIN_DIGITOS_DOCUMENTO, F.lit("documento_incompleto"))
    .when(F.col("doc_digitos").rlike(r"^(\d)\1*$"), F.lit("documento_sentinela_repetido"))
)

df_doc = (
    df_dedup
    .withColumn("doc_digitos", doc_digitos)
    .withColumn("motivo_rejeicao", motivo_pos)
)

df_rej_pos = df_doc.filter(F.col("motivo_rejeicao").isNotNull())
df_ok = df_doc.filter(F.col("motivo_rejeicao").isNull())

# Quarentena: colunas originais da Bronze + motivo + timestamp
cols_bronze = df_bronze.columns
df_quarentena = (
    df_rej_pre.select(*cols_bronze, "motivo_rejeicao")
    .unionByName(df_rej_pos.select(*cols_bronze, "motivo_rejeicao"))
    .withColumn("quarentena_datetime", F.current_timestamp())
)
print("Validacao de documentos e quarentena definidas (lazy).")

In [ ]:
# ==============================================================================
# CELULA 5: Padronizacao, tipagem e metadados da Silver
# ==============================================================================
# Documento: so digitos; <= 11 vira CPF (lpad 11), senao CNPJ (lpad 14)
doc_padronizado = (
    F.when(F.length("doc_digitos") <= 11, F.lpad("doc_digitos", 11, "0"))
    .otherwise(F.lpad("doc_digitos", 14, "0"))
)

df_silver = (
    df_ok
    # Identificacao e titular
    .withColumn("id_uc", F.col("id_uc_norm"))
    .withColumn("nome_titular", F.col("nome_norm"))

    # Documento
    .withColumn("cpf_cnpj", doc_padronizado)
    .withColumn("tipo_documento", F.when(F.length("cpf_cnpj") == 11, "CPF").otherwise("CNPJ"))
    .withColumn("documento_dv_valido", udf_documento_dv_valido(F.col("cpf_cnpj")))

    # Endereco
    .withColumn("logradouro", limpar_espacos(F.col("logradouro")))
    .withColumn("numero", F.trim(F.col("numero")))
    .withColumn("bairro", texto_padrao(F.col("bairro")))
    .withColumn("municipio", texto_padrao(F.col("municipio")))

    # UF: remove pontos/espacos (P.E. -> PE), nome por extenso de PE, dominio das 27 UFs
    .withColumn("_uf", F.regexp_replace(F.upper(F.trim(F.col("uf"))), r"[\.\s]", ""))
    .withColumn("_uf", F.when(F.col("_uf") == "PERNAMBUCO", F.lit("PE")).otherwise(F.col("_uf")))
    .withColumn("uf", F.when(F.col("_uf").isin(*UFS_VALIDAS), F.col("_uf")))

    # CEP: so digitos, exatamente 8, sem sentinelas (99991231, 00000000) nem digitos repetidos
    .withColumn("_cep", F.regexp_replace(F.coalesce(F.col("cep"), F.lit("")), r"\D", ""))
    .withColumn(
        "cep",
        F.when(
            (F.length("_cep") == 8)
            & (~F.col("_cep").rlike(r"^(\d)\1*$"))
            & (~F.col("_cep").isin(*CEPS_SENTINELA)),
            F.col("_cep"),
        ),
    )

    # Classificacao
    .withColumn("classe", texto_padrao(F.col("classe")))
    .withColumn("subgrupo_tarifario", F.upper(limpar_espacos(F.col("subgrupo_tarifario"))))
    .withColumn("alimentador", F.upper(limpar_espacos(F.col("alimentador"))))
    .withColumn("status", F.upper(limpar_espacos(F.col("status"))))

    # Data (ja tipada e restrita na celula 3)
    .withColumn("data_ligacao", F.col("data_ligacao_dt"))

    # Coordenadas: virgula decimal aceita; par fora da caixa do Brasil vira NULL/NULL
    .withColumn("_lat", F.expr("try_cast(replace(trim(latitude), ',', '.') as double)"))
    .withColumn("_lon", F.expr("try_cast(replace(trim(longitude), ',', '.') as double)"))
    .withColumn(
        "_coord_ok",
        F.col("_lat").between(BRASIL_LAT_MIN, BRASIL_LAT_MAX)
        & F.col("_lon").between(BRASIL_LON_MIN, BRASIL_LON_MAX),
    )
    .withColumn("latitude", F.when(F.col("_coord_ok"), F.col("_lat")))
    .withColumn("longitude", F.when(F.col("_coord_ok"), F.col("_lon")))

    # Metadado Silver (ingestion_datetime e source_file vem da Bronze)
    .withColumn("silver_processed_datetime", F.current_timestamp())
)

colunas_silver = [
    "id_uc", "cpf_cnpj", "tipo_documento", "documento_dv_valido", "nome_titular",
    "logradouro", "numero", "bairro", "municipio", "uf", "cep",
    "classe", "subgrupo_tarifario", "alimentador", "data_ligacao", "status",
    "latitude", "longitude",
    "ingestion_datetime", "source_file", "silver_processed_datetime",
]
df_silver = df_silver.select(*colunas_silver)

display(df_silver.limit(10))

In [ ]:
# ==============================================================================
# CELULA 6: Gravacao da Silver e da Quarentena (Delta)
# ==============================================================================
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)
print(f"Silver gravada: {silver_table}")

(
    df_quarentena.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(quarantine_table)
)
print(f"Quarentena gravada: {quarantine_table}")

# Constraints e comentario (idempotentes). Falha aqui gera aviso; o gate real e a celula 7.
try:
    for coluna in ["id_uc", "cpf_cnpj", "nome_titular"]:
        spark.sql(f"ALTER TABLE {silver_table} ALTER COLUMN {coluna} SET NOT NULL")
    spark.sql(f"ALTER TABLE {silver_table} DROP CONSTRAINT IF EXISTS chk_cpf_cnpj_tamanho")
    spark.sql(
        f"ALTER TABLE {silver_table} "
        f"ADD CONSTRAINT chk_cpf_cnpj_tamanho CHECK (length(cpf_cnpj) IN (11, 14))"
    )
    spark.sql(
        f"COMMENT ON TABLE {silver_table} IS "
        f"'Silver de Unidades Consumidoras: 1 linha por id_uc, higienizada e tipada. Rejeitados em {quarantine_table}.'"
    )
    print("Constraints aplicadas (NOT NULL em id_uc, cpf_cnpj, nome_titular; CHECK de tamanho do documento).")
except Exception as e:
    print(f"AVISO: nao foi possivel aplicar constraints/comentario: {e}")

In [ ]:
# ==============================================================================
# CELULA 7: Reconciliacao de volumes e validacoes BLOQUEANTES (sobre as tabelas gravadas)
# ==============================================================================
falhas = []

def checar(nome, condicao, detalhe=""):
    print(f"[{'OK   ' if condicao else 'FALHA'}] {nome} {detalhe}")
    if not condicao:
        falhas.append(nome)

df_silver_final = spark.table(silver_table)
df_quar_final = spark.table(quarantine_table)

# --- Reconciliacao: Bronze = rejeitados_pre + duplicatas_descartadas + rejeitados_pos + Silver
n_silver = df_silver_final.count()
rej = {
    r["motivo_rejeicao"]: r["qtd"]
    for r in df_quar_final.groupBy("motivo_rejeicao").agg(F.count(F.lit(1)).alias("qtd")).collect()
}
n_rej_pre = sum(rej.get(m, 0) for m in MOTIVOS_PRE)
n_rej_pos = sum(rej.get(m, 0) for m in MOTIVOS_POS)

n_valid_pre = df_valid_pre.count()
n_ids_distintos = df_valid_pre.select("id_uc_norm").distinct().count()
n_duplicatas = n_valid_pre - n_ids_distintos

print("\n=== Reconciliacao de volumes ===")
print(f"Bronze (total lido)                      : {fmt(total_bronze)}")
for m in MOTIVOS_PRE:
    print(f"  (-) rejeitado: {m:<30}: {fmt(rej.get(m, 0))}")
print(f"  (-) duplicatas descartadas (dedup)     : {fmt(n_duplicatas)}")
for m in MOTIVOS_POS:
    print(f"  (-) rejeitado: {m:<30}: {fmt(rej.get(m, 0))}")
print(f"Silver (gravada)                         : {fmt(n_silver)}\n")

checar("motivos de quarentena conhecidos", set(rej) <= set(MOTIVOS_PRE + MOTIVOS_POS), f"{sorted(rej)}")
checar("pre-rejeicao + validos = Bronze", n_rej_pre + n_valid_pre == total_bronze)
checar("ids distintos validos = Silver + rejeitados de documento", n_ids_distintos == n_silver + n_rej_pos)
checar(
    "Bronze = rejeitados + duplicatas + Silver",
    total_bronze == n_rej_pre + n_duplicatas + n_rej_pos + n_silver,
)

# --- Invariantes estruturais da Silver
g = df_silver_final.agg(
    F.count(F.lit(1)).alias("total"),
    F.countDistinct("id_uc").alias("ids_distintos"),
    F.sum(F.when(F.col("id_uc").isNull(), 1).otherwise(0)).alias("nulos_id_uc"),
    F.sum(F.when(F.col("cpf_cnpj").isNull(), 1).otherwise(0)).alias("nulos_cpf_cnpj"),
    F.sum(F.when(F.col("nome_titular").isNull(), 1).otherwise(0)).alias("nulos_nome"),
    F.sum(F.when(~F.length("cpf_cnpj").isin(11, 14), 1).otherwise(0)).alias("doc_tamanho_invalido"),
    F.sum(F.when((F.col("tipo_documento") == "CPF") != (F.length("cpf_cnpj") == 11), 1).otherwise(0)).alias("tipo_inconsistente"),
    F.sum(F.when(F.col("cep").isNotNull() & ((F.length("cep") != 8) | F.col("cep").isin(*CEPS_SENTINELA)), 1).otherwise(0)).alias("cep_invalido"),
    F.sum(F.when(F.col("uf").isNotNull() & ~F.col("uf").isin(*UFS_VALIDAS), 1).otherwise(0)).alias("uf_invalida"),
    F.sum(F.when(F.col("id_uc").startswith("TESTE"), 1).otherwise(0)).alias("ids_teste"),
).collect()[0]

checar("Silver nao vazia", g["total"] > 0)
checar("id_uc unico", g["total"] == g["ids_distintos"], f"(total={fmt(g['total'])}, distintos={fmt(g['ids_distintos'])})")
checar("sem nulos em id_uc", (g["nulos_id_uc"] or 0) == 0)
checar("sem nulos em cpf_cnpj", (g["nulos_cpf_cnpj"] or 0) == 0)
checar("sem nulos em nome_titular", (g["nulos_nome"] or 0) == 0)
checar("cpf_cnpj com 11 ou 14 digitos", (g["doc_tamanho_invalido"] or 0) == 0)
checar("tipo_documento consistente com o tamanho", (g["tipo_inconsistente"] or 0) == 0)
checar("cep valido ou NULL", (g["cep_invalido"] or 0) == 0)
checar("uf valida ou NULL", (g["uf_invalida"] or 0) == 0)
checar("sem id_uc de teste", (g["ids_teste"] or 0) == 0)

assert not falhas, f"Validacoes bloqueantes falharam: {falhas}"
print("\nTodas as validacoes bloqueantes passaram.")

In [ ]:
# ==============================================================================
# CELULA 8: Perfilamento e validacoes de ALERTA (nao bloqueiam o pipeline)
# ==============================================================================
df_s = spark.table(silver_table)
total = df_s.count()

# 8.1 Nulos em colunas opcionais (apos as regras de saneamento)
print("=== Nulos em colunas opcionais ===")
for c in ["data_ligacao", "latitude", "longitude", "cep", "uf", "classe", "municipio", "bairro",
          "logradouro", "status", "alimentador", "subgrupo_tarifario"]:
    n = df_s.filter(F.col(c).isNull()).count()
    marca = "ALERTA" if n > 0 else "ok    "
    print(f"[{marca}] {c:<20}: {fmt(n)} ({(100.0 * n / total):.2f}%)")

# 8.2 Digito verificador de CPF/CNPJ (nao bloqueante: a base pode ser sintetica/anonimizada)
print("\n=== Digito verificador de CPF/CNPJ ===")
display(df_s.groupBy("tipo_documento", "documento_dv_valido").count().orderBy("tipo_documento", "documento_dv_valido"))

# 8.3 Dominios (para decidir listas de valores validos)
print("\n=== Dominios observados ===")
for c in ["status", "classe", "uf", "subgrupo_tarifario"]:
    print(f"-- {c}")
    df_s.groupBy(c).count().orderBy(F.col("count").desc()).show(30, truncate=False)

# 8.4 Nomes suspeitos (digitos so sao suspeitos para CPF; razao social pode ter numeros)
nomes = df_s.select("id_uc", "cpf_cnpj", "tipo_documento", "nome_titular")
susp = nomes.filter(
    (F.length("nome_titular") <= 3)
    | (~F.col("nome_titular").rlike(" "))
    | ((F.col("tipo_documento") == "CPF") & F.col("nome_titular").rlike("[0-9]"))
)
n_susp = susp.count()
print(f"\nNomes potencialmente suspeitos: {fmt(n_susp)}")
if n_susp > 0:
    display(susp.orderBy(F.length("nome_titular")).limit(20))

# 8.5 Informativo: documentos com mais de uma UC (titular com varias UCs e comum em energia)
n_docs = df_s.select("cpf_cnpj").distinct().count()
n_docs_multi = (
    df_s.groupBy("cpf_cnpj").agg(F.count(F.lit(1)).alias("qtd_ucs")).filter(F.col("qtd_ucs") > 1).count()
)
print(f"\nUCs: {fmt(total)} | documentos distintos: {fmt(n_docs)} | documentos com mais de uma UC: {fmt(n_docs_multi)}")
if n_docs_multi == 0:
    print("Relacao 1:1 entre UC e documento. Para uma distribuidora real isso e atipico: confirmar se a base e sintetica.")

# 8.6 Quarentena por motivo
print("\n=== Quarentena por motivo ===")
spark.table(quarantine_table).groupBy("motivo_rejeicao").count().orderBy(F.col("count").desc()).show(truncate=False)